
OG GPT architecture for reference.

1) tokenization,
tokenized text enters the model
|
v
---
2) token embedding layer: (this is the first step of our model - makes embeddings from text)

3) positional embeddding layer: (projects positional encoding onto token embeddings to get an understanding of whole sentence)

4) dropout: (dropout is a method to combat overfitting where nodes in a neural network are randomly dropped (removed) from the network to introduce sparsity. This was found to help the performance of neural networks and is very important. 

---
og calls for 48 of these blocks stacked 
---

5) layerNorm 1: this restricts the scale of the layer to a constant shape. We have a lot of matrix multiplication and layer norms keep shapes consistent and playing well together. 

6) masked multi head attention - 25 heads per block

7) dropout: same as above

8) layer norm 2

9) feed forward neural network: Like our paper, this is a standard feed forward nn whose hidden layer dimensions are 6400.
---

9a) Linear layer (affine transformation)

9b) GELU activation: 

9c) Linear Layer (affine transformation)

---
---

10) dropout

11) final layer norm

12) linear output



-forward pass they use attention to predict next token


---


For our model to keep it trainable we will use 5 blocks each with 8 heads per block. The FFN hidden dimension will be 512 (4 x D_model) the d model will be 128..
context lenght is 64-128

1 million params.

vocab is 65-100


A way less gnarly idea would be to implement a transformer with just the encoder (no decoder) and a single attention block in numpy. 

This looks conceptually similar from the above one minus the many stacking layers and last part

### What im actually making (frfr)
---

Im making a micro gpt with its own proprietary rust tensor library called clem

This is a 2 block (each with a single head of attention) transformer that has an encoder, ff neural network and predicts the next token. For inference, beam search would be cool to implement but thats really ambitious so i'll probably just try and keep it simple stupid.

Going to try and train on CPU so i'm going to do my best to write some rust that's callable via python in order to minimize any extra training overhead. 

In [ ]:
MAX_LENGTH = 256
BATCH_SIZE = 32 # lets be nice to our cpu.

class Tokenizer:
    def __init__(self, batches: list[list[str]]):
        """
        Each batch is a list of split sentences from paragraphs. 
        TODO: Consider writing text splitting module in rust and calling from 
        
        """
        self.idx_to_token = {}
        self.token_to_idx = {}
        self.MAX_LENGTH = MAX_LENGTH
        self.BATCH_SIZE = BATCH_SIZE
        # End of text is the core special token we use.
        # In the core gpt 2 this was used for padding, unknown tokens, end of sentence, etc.
        # newer models use more sophisticated means of tokenization (with bytes) but we are keeping things stupid simple.
        self.special_token = '<|endoftext|>'
        unique_tokens = set()
        for token in sentence.split(' ') for sentence in batch for batch in batches:
            unique_tokens.add(token)
        
        sorted_tokens = sorted(list(unique_tokens))
        
        idx_to_token = {index: token for index, token in enumerate(sorted_tokens)}
        token_to_idx = {token: index for index, token in enumerate(sorted_tokens)}
        
        self.idx_to_token = idx_to_token
        self.token_to_idx = token_to_idx
        self.vocab_size = unique_tokens.size()
        self.idx_to_token[len(unique_tokens)] = self.special_token
        self.token_to_idx[self.special_token] = len(unique_tokens)
        
    def decode(self, index):
        """
        Take a token, if it doesnt exist return special in leu but output a response
        """
        return self.idx_to_token.get(index, self.special_token)
    
    def encode(self, token):
        """
        Inverse of above!
        """
        if token not in self.token_to_idx:
            return self.token_to_idx[self.special_token]
        return self.token_to_idx[token]
    
    def input_query_to_embedding_with_padding(self, sentence):
        """
        Used when training!
        """
        tokens = sentence.split(' ')
        
        tokens = tokens[0:self.MAX_LENGTH]
        
        if len(tokens) < self.MAX_LENGTH:
            tokens.extend([self.special_token for i in range(self.MAX_LENGTH - len(tokens))]
        
        vector = [self.encode(token) for token in tokens]
        
        return vector

    def stack_embeddings_into_tensor(self, sentence_batch:list[str]):
        padded_embeddings = [self.input_query_to_embedding_with_padding(sent) for sent in sentence_batch]
        # Clem is our hand rolled torch alternative for training, the dtype here defaults to float 32.
        tensor = clem.tensor((padded_embeddings, self.BATCH_SIZE))
        assert tensor.shape == (32, 256)
        return tensor
        

In [ ]:
class Encoder:
    """
    Map the tokenized text we are training on into a vocab space which is an indexed lookup table 
    comprised of [index:int]: [value: word]
    Remember our tokenizer takes a sentence and converts it to a vector of integers which correspond to the token in our vocabulary.
    
    Next we project a positional encoding onto our sentence encodings. 
    
    we need d_model - which is our hidden dimension size. 
    for 2 blocks each containing one attention layer, we will pick an arbitray d_model that satisifies the constraints with respect to num heads
    d-model - 256
    n heads - 2
    d_head = d-model / n heads = 128

    Vocab size is derived from the tokenizer as an attribute defined in the constructor
    """
    def __init__(
        self, 
        tokenizer:Tokenizer,
        vocab_size:int, 
        num_heads:int=2,
        d_model:int = 128,
        max_seq_len:int = 256,
        max_batch_size:int=32
    ):
        self.tokenizer = tokenizer
        self.vocab_size = tokenizer.vocab_size
        assert self.vocab_size is not None

        self.num_heads = num_heads
        self.d_model = d_model # hidden dimmensions of our model.
        # initialize random embeddings - these become learned later so randn to start is normal.
        self.embedding_table = clem.randn(self.vocab_size, self.d_model)
        self.max_seq_len = max_seq_len # max seq length of our model - 256.
        self.max_batch_size = max_batch_size # 32 for CPU friendly ness. 
        self.d_head = self.d_model / self.num_heads 
        assert d_head
    
    def create_embeddings_from_token_idxs(self, token_idx_tensor:clem.tensor) -> clem.tensor:
        # Convert our indices into our randomly initialized embeddings via embedding table. 
        return self.embedding_table[token_idx_tensor]

    def positionally_encode_embeddings(self, embeddings:clem.Tensor):
        # re-scale embeddings through element wise multiplication by the square root of d_model.
        # When we project the positional encoding of sentences onto our token embeddings, we are 
        # Compressing information into a singular tensor. This has the capacity to drown out the initial meaning of each token,
        # with all the new postiional information we are projecting. 

        # This first step lifts up the semantic meaning of our token embeddings to prevent their meaning from being lost when
        # positional embeddings are later projected onto them. 
        e_prime = embeddings * math.sqrt(self.d_model)
        # Start at 0 - go till d_model spaced by 2.
        i = clem.arange(0, self.d_model, 2)
        # e'{j^2} element wise exponential multiplication by -i / self.d_model * log (tensor)
        # POS is a column vector of shape (max_seq_len, 1) so it can broadcast against div_term 
        # Broadcasting is when you have two tensors with different shapes but it is still possible to 
        # multiply them together. 
        # e.g.: 
        pos = clem.arange(self.max_seq_len, dtype='float32').reshape(self.max_seq_len, 1)

        # this gives us our even indices we can use while filling our positional encoding with data.
        div_term =  clem.exp(-i / self.d_model * clem.log(clem.tensor(10000.0)))
        
        positional_encoding = clem.zeros(self.max_seq_len, self.d_model)
        positional_encoding[:, 0::2] = clem.sin(pos * div_term)
        positional_encoding[:, 1::2] = clem.cos(pos * div_term)
        
        sliced_pos = positional_encoding[:embeddings.shape[1], :]
        return sliced_pos + e_prime
        
        

In [ ]:
class SingleAttentionHeadBlock:
    """
    TODO: Figure out what goes into this. conceptually. We need a tensor wrapper, we understand that this is comprised of 
    attention which is a projection of query, key, value across multi dimensional space, including some masking and normalization steps too 
    (softmax) for making prediction. We are going to use 2 of these in our model.
    """
    def __init__(self, *args):
        pass
        

In [ ]:
class FeedForwardNeuralNetwork:
    """
    Takes the tensor from our singleAttentionHeadBlock and approximates what the next 
    """

In [ ]:
class MicroGPT:
    """
    Really small - like clem
    """
    def __init__(self, tokenizer, encoder):
        self.tokenizer = tokenizer
        self.encoder = encoder

In [ ]:
with open("corpus.txt", "r", encoding="utf-8") as file:
    corpus = file.read()

all_sentences = corpus.split('.')
num_batches = math.ceil(len(all_sentences) / BATCH_SIZE)
batches = []

for i in range(num_batches):
    start = i * BATCH_SIZE
    end = min(i + 1 * BATCH_SIZE, num_batches)
    
    batches.append(all_sentences[start:end])

tokenizer = Tokenizer(batches)
encoder = Encoder(tokenizer)
model = MicroGPT(tokenizer)

# Training on CPU for starters. 
for epoch in epochs:
    for sentence_batch in batches:
        # start training by assembling the correct shape of our batched training data. 
        idx_tensor = model.tokenizer.stack_token_idx_into_tensor(sentence_batch)
        embeddings = model.encoder.create_embeddings_from_token_idxs(idx_tensor)
        # project positional embeddings onto 
        model.encoder.positionally_encode_embeddings(tensor)